In [2]:
! pip install bertopic sentence-transformers pandas numpy scikit-learn matplotlib seaborn --break-system-packages


Defaulting to user installation because normal site-packages is not writeable
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.1/7.1 MB 13.4 MB/s  0:00:00 eta 0:00:01
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 13.9 MB/s  0:00:00m0:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 5.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╺━━━━━━━━ 3.1/4.0 MB 13.8 MB/s eta 0:00:01
Resuming download hf_xet-1.5.2-cp38-abi3-macosx_10_12_x86_64.whl (3.1 MB/4.0 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.0/4.0 MB 7.5 MB/s  0:00:00-:--:--
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 12.6 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.1/31.1 MB 14.3 MB/s  0:00:02 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 13.9 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [4]:
! pip install "numpy<2" --break-system-packages --force-reinstall

Defaulting to user installation because normal site-packages is not writeable
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.6/20.6 MB 32.5 MB/s  0:00:00m0:00:0100:01
  Attempting uninstall: numpy
    Found existing installation: numpy 2.0.2
    Uninstalling numpy-2.0.2:
      Successfully uninstalled numpy-2.0.2


In [5]:
"""
BERTOPIC CLUSTER ANALYSIS ON COMMENT TEXT
=============================================
Goal: let topics emerge from the actual comment text (unsupervised),
as a complement to / sanity check on the planned House-typology LLM
annotation. Surfaces WHAT people talk about, not just support-type
structure.

Design notes:
    - Uses 'body' (original text), NOT 'cleaned_body' — the cleaning
      pipeline replaced URLs with the literal token "URL" and stripped
      punctuation/casing, which would pollute topics with a meaningless
      "url" cluster and hurts sentence-embedding quality (embeddings
      do better on natural, punctuated text).
    - Does its own lighter cleaning pass instead (strip URLs entirely,
      strip Reddit markdown, keep sentence structure/casing).
    - Filters out bot accounts (AutoModerator etc.) — template comments
      form spurious "topics" that are just repeated boilerplate.
    - Embeddings are checkpointed to .npy — re-running with different
      BERTopic settings (nr_topics, min_topic_size) won't require
      re-embedding ~55K comments from scratch.

Install once:
    pip install bertopic sentence-transformers pandas numpy scikit-learn \
        matplotlib seaborn --break-system-packages

Reads:
    master_comments_filtered.csv

Writes:
    comment_topics_assigned.csv        (every comment + its topic)
    comment_topic_info.csv              (topic sizes, top words per topic)
    comment_topics_by_batscore.csv      (cross-tab: topic distribution by bat_score)
    bertopic_model/                     (saved model, for reuse without refitting)
    comment_embeddings.npy              (checkpointed embeddings)
    topic_size_barplot.png
"""

import pandas as pd
import numpy as np
import re
import os
import warnings
warnings.filterwarnings('ignore')

from bertopic import BERTopic
from sentence_transformers import SentenceTransformer
import matplotlib.pyplot as plt
import seaborn as sns

# ── CONFIG ───────────────────────────────────────────────────────────────────
DATA_DIR = '/Users/nadia/Desktop/redditRun_june/comment_data/'
INPUT_FILE = DATA_DIR + 'master_comments_filtered.csv'
OUTPUT_DIR = DATA_DIR

EMBEDDING_CHECKPOINT = OUTPUT_DIR + 'comment_embeddings.npy'
MODEL_SAVE_DIR = OUTPUT_DIR + 'bertopic_model'
EMBEDDING_MODEL_NAME = 'all-MiniLM-L6-v2'   # fast, good quality, runs fine on CPU

EXCLUDE_AUTHORS = ['AutoModerator', '[deleted]']

MIN_TOPIC_SIZE = 30      # smaller = more, finer-grained topics
NR_TOPICS = 'auto'       # let BERTopic decide, or set an int to force a target count

RANDOM_STATE = 42
sns.set_style("whitegrid")


# ── STEP 1: LOAD + LIGHT CLEAN (separate from the word-count-filter cleaning) ─
def clean_for_bertopic(text):
    if pd.isna(text):
        return ""
    text = re.sub(r'http\S+|www\S+', ' ', text)                    # strip URLs entirely
    text = re.sub(r'\[([^\]]+)\]\([^\)]+\)', r'\1', text)          # markdown links -> just the text
    text = re.sub(r'[*_~`>#]+', '', text)                          # strip markdown formatting chars
    text = re.sub(r'&amp;', '&', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text


def load_data():
    df = pd.read_csv(INPUT_FILE)
    print(f"Loaded {len(df):,} comments")

    before = len(df)
    df = df[~df['author'].isin(EXCLUDE_AUTHORS)]
    print(f"Removed {before - len(df):,} bot/deleted-author comments")

    df['bertopic_text'] = df['body'].apply(clean_for_bertopic)

    before = len(df)
    df = df[df['bertopic_text'].str.len() > 0]
    print(f"Removed {before - len(df):,} comments empty after cleaning")

    df = df.reset_index(drop=True)
    print(f"Final corpus for topic modeling: {len(df):,} comments")
    return df


# ── STEP 2: EMBEDDINGS (checkpointed) ───────────────────────────────────────
def get_embeddings(docs):
    if os.path.exists(EMBEDDING_CHECKPOINT):
        embeddings = np.load(EMBEDDING_CHECKPOINT)
        if embeddings.shape[0] == len(docs):
            print(f"Loaded cached embeddings: {embeddings.shape}")
            return embeddings
        else:
            print(f"⚠ Cached embeddings shape {embeddings.shape} doesn't match "
                  f"current doc count {len(docs)} — recomputing")

    print(f"Computing embeddings with {EMBEDDING_MODEL_NAME}... "
          f"(this is the slow step, ~a few minutes for ~50K docs on CPU)")
    model = SentenceTransformer(EMBEDDING_MODEL_NAME)
    embeddings = model.encode(docs, show_progress_bar=True, batch_size=64)
    np.save(EMBEDDING_CHECKPOINT, embeddings)
    print(f"✓ Embeddings cached to {EMBEDDING_CHECKPOINT}")
    return embeddings


# ── STEP 3: FIT BERTOPIC ────────────────────────────────────────────────────
def fit_topic_model(docs, embeddings):
    topic_model = BERTopic(
        embedding_model=EMBEDDING_MODEL_NAME,
        min_topic_size=MIN_TOPIC_SIZE,
        nr_topics=NR_TOPICS,
        calculate_probabilities=False,   # faster; set True if you want per-doc topic probability distributions
        verbose=True,
    )
    topics, _ = topic_model.fit_transform(docs, embeddings)
    return topic_model, topics


# ── STEP 4: SUMMARIZE ────────────────────────────────────────────────────────
def summarize_topics(topic_model, df, topics):
    df['topic'] = topics

    topic_info = topic_model.get_topic_info()
    print("\n" + "=" * 70)
    print(f"FOUND {len(topic_info) - 1} TOPICS (excluding -1 = outliers/no clear topic)")
    print("=" * 70)
    print(topic_info[['Topic', 'Count', 'Name']].to_string(index=False))

    n_outliers = (df['topic'] == -1).sum()
    print(f"\nOutlier comments (topic=-1, didn't fit any cluster): "
          f"{n_outliers:,} ({n_outliers/len(df)*100:.1f}%)")

    print("\n" + "─" * 70)
    print("TOP 15 TOPICS — keywords + example comments")
    print("─" * 70)
    top_topics = topic_info[topic_info['Topic'] != -1].head(15)
    for _, row in top_topics.iterrows():
        topic_id = row['Topic']
        words = [w for w, _ in topic_model.get_topic(topic_id)][:8]
        examples = df.loc[df['topic'] == topic_id, 'body'].head(2).tolist()
        print(f"\nTopic {topic_id} (n={row['Count']}): {', '.join(words)}")
        for ex in examples:
            snippet = ex[:150].replace('\n', ' ')
            print(f"    e.g. \"{snippet}...\"")

    return topic_info


# ── STEP 5: CROSS-TAB WITH bat_score ────────────────────────────────────────
def crosstab_with_batscore(df, topic_info):
    print("\n" + "─" * 70)
    print("TOPIC DISTRIBUTION BY bat_score")
    print("─" * 70)

    if 'bat_score' not in df.columns:
        print("⚠ bat_score column not found in input file — skipping cross-tab")
        return None

    crosstab = pd.crosstab(df['topic'], df['bat_score'], normalize='columns') * 100
    top_topic_ids = topic_info[topic_info['Topic'] != -1].head(15)['Topic'].tolist()
    print(crosstab.loc[crosstab.index.isin(top_topic_ids)].round(1))

    return crosstab


# ── STEP 6: VISUALIZATION ───────────────────────────────────────────────────
def make_figure(topic_info):
    top20 = topic_info[topic_info['Topic'] != -1].head(20)
    fig, ax = plt.subplots(figsize=(10, 8))
    ax.barh(top20['Name'], top20['Count'], color='mediumseagreen', edgecolor='black')
    ax.set_xlabel('Comment count')
    ax.set_title('Top 20 Comment Topics by Size')
    ax.invert_yaxis()
    plt.tight_layout()
    out_path = OUTPUT_DIR + 'topic_size_barplot.png'
    plt.savefig(out_path, dpi=200, bbox_inches='tight')
    print(f"\n✓ Figure saved: {out_path}")


# ── MAIN ─────────────────────────────────────────────────────────────────────
if __name__ == '__main__':
    print("=" * 80)
    print("BERTOPIC CLUSTER ANALYSIS ON COMMENT TEXT")
    print("=" * 80)

    df = load_data()
    docs = df['bertopic_text'].tolist()

    embeddings = get_embeddings(docs)
    topic_model, topics = fit_topic_model(docs, embeddings)
    topic_info = summarize_topics(topic_model, df, topics)
    crosstab = crosstab_with_batscore(df, topic_info)
    make_figure(topic_info)

    # Save everything
    topic_model.save(MODEL_SAVE_DIR, serialization="safetensors",
                      save_ctfidf=True, save_embedding_model=EMBEDDING_MODEL_NAME)
    print(f"\n✓ Model saved to {MODEL_SAVE_DIR}")

    out_cols = ['id', 'post_id', 'bat_score', 'author', 'body', 'topic', 'word_count']
    out_cols = [c for c in out_cols if c in df.columns]
    df[out_cols].to_csv(OUTPUT_DIR + 'comment_topics_assigned.csv', index=False)
    print(f"✓ comment_topics_assigned.csv saved  →  {len(df):,} comments")

    topic_info.to_csv(OUTPUT_DIR + 'comment_topic_info.csv', index=False)
    print(f"✓ comment_topic_info.csv saved  →  {len(topic_info):,} topics")

    if crosstab is not None:
        crosstab.to_csv(OUTPUT_DIR + 'comment_topics_by_batscore.csv')
        print(f"✓ comment_topics_by_batscore.csv saved")

BERTOPIC CLUSTER ANALYSIS ON COMMENT TEXT
Loaded 54,935 comments
Removed 326 bot/deleted-author comments
Removed 0 comments empty after cleaning
Final corpus for topic modeling: 54,609 comments
Computing embeddings with all-MiniLM-L6-v2... (this is the slow step, ~a few minutes for ~50K docs on CPU)


Batches: 100%|██████████| 854/854 [45:23<00:00,  3.19s/it]  


RuntimeError: Numpy is not available